# Dataset Creation


## Imports


In [1]:
import os
import shutil
import subprocess
import xml.etree.ElementTree as ET

import numpy as np

## Constants


In [2]:
# Set random seed for reproducibility
np.random.seed(42)

# Set SUMO_HOME environment variable
os.environ["SUMO_HOME"] = os.path.join(os.environ["CONDA_PREFIX"], "Lib", "site-packages", "sumo")
SUMO_HOME = os.environ["SUMO_HOME"]

# Add SUMO_HOME/bin and SUMO_HOME/tools to PATH
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "bin")
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "tools")

# Directories for route files
ROUTES_TRAIN_DIR = "routes_train"
ROUTES_TEST_DIR = "routes_test"
ROUTES_FAILED_DIR = "routes_failed"

# Create directories if they do not exist
for dir in [ROUTES_TRAIN_DIR, ROUTES_TEST_DIR, ROUTES_FAILED_DIR]:
    if not os.path.exists(dir):
        os.makedirs(dir)

# Paths to network and tools
NETWORK_PATH = os.path.join("osmWebWizard", "osm.net.xml.gz")
RANDOM_TRIPS_PATH = os.path.join(SUMO_HOME, "tools", "randomTrips.py")
DUAROUTER_PATH = os.path.join(SUMO_HOME, "bin", "duarouter.exe")

# Vehicle types and their probabilities
VEHICLE_TYPES_PROBABILITIES = {"car": 0.70, "ldv": 0.10, "truck": 0.05, "bus": 0.15}

# Define traffic generation volumes for 10 hours of traffic
RAW_TRAFFIC_VOLUMES = [1, 2, 6, 5, 3, 1, 2, 6, 4, 3]

## Traffic Volumes


In [3]:
# Normalize and scale raw volumes
traffic_volumes = [12 / x for x in RAW_TRAFFIC_VOLUMES]

# Generate train volumes
train_traffic_volumes = list(np.array(traffic_volumes))

# Generate test traffic volumes based on train volumes multiplied by a small random factor
test_traffic_volumes = [v * np.random.uniform(0.95, 1.05) for v in train_traffic_volumes]

## Random Trips


In [ ]:
def generate_random_trips(traffic_volumes, output_directory):
    for hour, volume in enumerate(traffic_volumes):
        start_time = hour * 3600
        end_time = start_time + 3600
        output_file = os.path.join(output_directory, f"hour_{hour:02d}.rou.xml")
        command = [
            "python",
            RANDOM_TRIPS_PATH,
            "-n",
            NETWORK_PATH,
            "-b",
            str(start_time),
            "-e",
            str(end_time),
            "-p",
            str(volume),
            "--edge-param",
            "main",
            "--validate",
            "-o",
            output_file,
        ]
        print("Executing:", " ".join(command))
        result = subprocess.run(command, capture_output=True, text=True)
        print(result.stdout)
        print(result.stderr)
        if os.path.exists(output_file):
            print("Created:", output_file)
        else:
            print("Failed:", output_file)


print("Generating train random trips...")
generate_random_trips(train_traffic_volumes, ROUTES_TRAIN_DIR)
print("Generating test random trips...")
generate_random_trips(test_traffic_volumes, ROUTES_TEST_DIR)

Generating train random trips routes...
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n osmWebWizard\osm.net.xml.gz -b 0 -e 3600 -p 12.0 --edge-param main --validate -o routes_train\hour_00.rou.xml
Success.
Success.


Created: routes_train\hour_00.rou.xml
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n osmWebWizard\osm.net.xml.gz -b 3600 -e 7200 -p 6.0 --edge-param main --validate -o routes_train\hour_01.rou.xml
Success.
Success.


Created: routes_train\hour_01.rou.xml
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n osmWebWizard\osm.net.xml.gz -b 7200 -e 10800 -p 2.0 --edge-param main --validate -o routes_train\hour_02.rou.xml
Success.
Success.


Created: routes_train\hour_02.rou.xml
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py -n osmWebWizard\osm.net.xml.gz -b 1080

## Duarouter


In [ ]:
# Step 2: Convert trip-format files to vehicle-format using duarouter
def run_duarouter(directory):
    for filename in sorted(os.listdir(directory)):
        if filename.endswith(".rou.xml"):
            trip_file = os.path.join(directory, filename)
            veh_file = os.path.join(directory, filename.replace(".rou.xml", ".veh.xml"))
            command = f'"{DUAROUTER_PATH}" -n {NETWORK_PATH} -r {trip_file} -o {veh_file}'
            print("Executing duarouter:", command)
            result = subprocess.run(command, shell=True, capture_output=True, text=True)
            print(result.stdout)
            print(result.stderr)
            if os.path.exists(veh_file):
                print("Converted to vehicle format:", veh_file)
            else:
                print("Conversion failed for:", trip_file)
                shutil.move(
                    trip_file,
                    os.path.join(ROUTES_FAILED_DIR, os.path.basename(trip_file)),
                )


print("Converting train routes from trip to vehicle format...")
run_duarouter(ROUTES_TRAIN_DIR)
print("Converting test routes from trip to vehicle format...")
run_duarouter(ROUTES_TEST_DIR)

In [ ]:
# Step 3: Update vehicle IDs in converted files (files ending with .veh.xml)
def update_vehicle_ids(directory, suffix=".veh.xml"):
    vehicle_id = 0
    for filename in sorted(os.listdir(directory)):
        if filename.endswith(suffix):
            file_path = os.path.join(directory, filename)
            tree = ET.parse(file_path)
            root = tree.getroot()
            for vehicle in root.findall("vehicle"):
                vehicle.set("id", f"veh{vehicle_id}")
                vehicle_id += 1
            tree.write(file_path)
            print("Updated IDs in:", file_path)


print("Updating vehicle IDs in train files...")
update_vehicle_ids(ROUTES_TRAIN_DIR)
print("Updating vehicle IDs in test files...")
update_vehicle_ids(ROUTES_TEST_DIR)

In [ ]:
def update_vehicle_types(directory, suffix=".veh.xml"):
    for filename in sorted(os.listdir(directory)):
        if filename.endswith(suffix):
            file_path = os.path.join(directory, filename)
            tree = ET.parse(file_path)
            root = tree.getroot()
            for vehicle in root.findall("vehicle"):
                chosen = np.random.choice(
                    list(VEHICLE_TYPES_PROBABILITIES.keys()),
                    p=list(VEHICLE_TYPES_PROBABILITIES.values()),
                )
                vehicle.set("type", chosen)  # 'type' attribute links to vtypes.xml definitions
            tree.write(file_path)
            print("Updated vehicle types in:", file_path)


print("Updating vehicle types in train files...")
update_vehicle_types(ROUTES_TRAIN_DIR)
print("Updating vehicle types in test files...")
update_vehicle_types(ROUTES_TEST_DIR)